In [1]:
from src.citibike.citibike_utils import get_trip_duration_mins, null_interpolation, timestamp_to_date
from pyspark.sql.functions import create_map, lit

In [ ]:
pipeline_id = dbutils.widgets.get("pipeline_id")
run_id = dbutils.widgets.get("run_id")
task_id = dbutils.widgets.get("task_id")
processed_timestamp = dbutils.widgets.get("processed_timestamp")
catalog = dbutils.widgets.get("catalog")

In [ ]:
df= spark.read.format("delta").table(f"{catalog}.bronze.jc_citybike")

In [3]:
df = null_interpolation(spark, df)

In [4]:
df = get_trip_duration_mins(spark, df, "started_at", "ended_at", "trip_duration_mins")

In [5]:
df = timestamp_to_date(spark, df, "started_at", "started_date")
df = timestamp_to_date(spark, df, "ended_at", "ended_date")

In [ ]:
df = df.withColumn("metadata", 
              create_map(
                  lit("pipeline_id"), lit(pipeline_id),
                  lit("run_id"), lit(run_id),
                  lit("task_id"), lit(task_id),
                  lit("processed_timestamp"), lit(processed_timestamp)
                  ))

In [7]:
df = df.select(
    "ride_id",
    "started_date",
    "started_at",
    "ended_date",
    "ended_at",
    "start_station_name",
    "end_station_name",
    "trip_duration_mins",
    "metadata"
    )

In [ ]:
df.write.format("delta").mode("overwrite").saveAsTable(f"{catalog}.silver.jc_citybike")

c:\Users\soham\OneDrive\Desktop\databricks_cicd\.venv\Lib\site-packages\pyspark\sql\connect\expressions.py:1134: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(
